# CSE 2600 HW 3

## Q1

A. 
\begin{gather*}
\log\left(\frac{p}{1 - p}\right) = \beta_0 + \beta_1 X_1 + \beta_2 X_2\\
\beta_0 = -4.2,\quad \beta_1 = 0.04,\quad \beta_2 = 0.8\\
z = \beta_0 + \beta_1 X_1 + \beta_2 X_2\\
z = -4.2 + 0.04(60) + 0.8(2)\\
z = -0.2\\
p = \frac{1}{1 + e^{-z}}\\
p = \frac{1}{1 + e^{0.2}} \approx 0.45
\end{gather*}
So the estimated probability is 0.45 or 45%.

B. 
\begin{gather*}
p=0.9\\
p = \frac{1}{1 + e^{-z}}\\
0.9 = \frac{1}{1 + e^{-z}}\\
1 + e^{-z} = \frac{1}{0.9}\\
1 + e^{-z} \approx 1.1111\\
e^{-z} = 0.1111\\
-z = ln(0.1111)\\
-z \approx -2.1972\\
z = 2.1972\\
2.1972 = −4.2+0.04X_1​+0.8(2)\\
2.1972 = -2.6+0.04X_1\\
4.7972 = 0.04X_1\\
X_1 = 119.93
\end{gather*}
So the applicant would need about 120 hours of practice.

C. 
$\beta_0$ is the intercept in the logistic regression. It represents the log-odds of admission when $X_1 = 0$ and $X_2 = 0$ (someone with 0 coding hours and 0 years of prior experience). Here, $-4.2$ is a very negative log-odds, meaning the probability of admission for such an applicant is very low. It odds would be $p = \frac{1}{1 + e^{-4.2}} \approx 0.0147$ or about $1.47\%.$

## Q2

In [29]:
from ISLP import load_data
import pandas as pd
import numpy as np
import statsmodels.api as sm

OJ = load_data('OJ')

Train = OJ[OJ['StoreID'].isin([1, 2, 3, 4])].copy()
Test = OJ[OJ['StoreID'] == 7].copy()

#Part A, Calculate mean, standard deviation, median, minimum value, and maximum value
Train['PriceDiff'] = Train['PriceMM'] - Train['PriceCH']
train_subset = Train[['LoyalCH', 'SpecialCH', 'PriceDiff']] 
summary_stats = train_subset.agg(['mean', 'std', 'median', 'min', 'max'])
print(f"\n\033[1mA.\033[0m")
print(summary_stats)

#Part B, Calculate the correlation matrix
correlation_matrix = train_subset.corr()
print(f"\n\033[1mB.\033[0m")
print(correlation_matrix)

#Part C, Perform a logistic regression
Train['Purchase_CH'] = (Train['Purchase'] == 'CH').astype(int)
X_train = Train[['LoyalCH', 'SpecialCH', 'PriceDiff']].copy()
X_train['intercept'] = 1  
y_train = Train['Purchase_CH']
glm = sm.GLM(y_train, X_train, family=sm.families.Binomial())
results = glm.fit()
print(f"\n\033[1mC.\033[0m")
print(results.summary())



A.
         LoyalCH  SpecialCH  PriceDiff
mean    0.519988   0.046218   0.204762
std     0.313757   0.210105   0.118882
median  0.500000   0.000000   0.240000
min     0.000011   0.000000   0.000000
max     0.999947   1.000000   0.440000

B.
            LoyalCH  SpecialCH  PriceDiff
LoyalCH    1.000000   0.014230   0.013981
SpecialCH  0.014230   1.000000  -0.115511
PriceDiff  0.013981  -0.115511   1.000000

C.
                 Generalized Linear Model Regression Results                  
Dep. Variable:            Purchase_CH   No. Observations:                  714
Model:                            GLM   Df Residuals:                      710
Model Family:                Binomial   Df Model:                            3
Link Function:                  Logit   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -310.82
Date:                Sun, 19 Oct 2025   Deviance:                       621.63
Time:                        23:1

**Cont. C.** Based on the results of the logistic regression, two of the three predictors seem to be statistically significant in predicting whether a customer purchases Citrus Hill. The variable LoyalCH has a coefficient of 6.118 with a p-value of 0.000, showing a highly significant positive effect: as loyalty increases, the probability of purchasing Citrus Hill rises sharply. Similarly, PriceDiff has a coefficient of 2.824 with a p-value of 0.001, showing that as Citrus Hill becomes cheaper relative to Minute Maid, the odds of purchase increase significantly. Whereas, SpecialCH has a coefficient near zero (-0.022) and a very high p-value (0.961), suggesting that the presence of a Citrus Hill promotion does not have a meaningful effect on purchase behavior. Therefore, LoyalCH and PriceDiff are statistically significant predictors, while SpecialCH is not.

In [ ]:
#Part D, compute the confusion matrix and overall fraction of correct predictions. 
Test['PriceDiff'] = Test['PriceMM'] - Test['PriceCH']
Test['Purchase_CH'] = (Test['Purchase'] == 'CH').astype(int)
X_test = Test[['LoyalCH','SpecialCH','PriceDiff']].copy()
X_test['intercept'] = 1
y_test = Test['Purchase_CH']
def predict(X, model):
    predictions_df = pd.DataFrame(model.get_prediction(X).predicted, columns=['y_hat'], index=X.index)
    return predictions_df['y_hat']
probs_train = predict(X_train, results)
probs_test = predict(X_test, results)
predictions_train = (probs_train >= 0.5).astype(int)
predictions_test = (probs_test >= 0.5).astype(int)


